## Transform Addresses Data
1. Create one record for each customer with 2 sets of address columns, 1 for shipping and 1 for billing address 
1. Write transformed data to the Silver schema  

In [0]:
SELECT customer_id,
       address_type,
       address_line_1,
       city,
       state,
       postcode
  FROM gizmo_box.bronze.v_addresses;

customer_id,address_type,address_line_1,city,state,postcode,_rescued_data
9179,shipping,9713 Medina Mountains Apt. 813,Lake Erika,Mississippi,91945,null
9179,billing,084 Anne Hollow Apt. 064,East Jasontown,Minnesota,77329,null
5816,shipping,8400 Rebecca Lodge Suite 011,Lake Lisatown,Montana,11354,null
5816,billing,869 Linda Locks Apt. 105,West Melissabury,Michigan,37662,null
4858,shipping,4838 Jacob Neck,Jamesview,Wisconsin,58657,null
4858,billing,4558 Cabrera Islands,Lake David,Rhode Island,6133,null
7207,shipping,75470 Richardson Mission Apt. 147,New Rachelhaven,Hawaii,38857,null
7207,billing,4450 Michelle Mission Suite 663,North Justin,Kentucky,91757,null
8539,shipping,331 Monica Route Apt. 022,Michellechester,Texas,14637,null
8539,billing,48794 Aguirre Ferry Apt. 952,North Amy,Washington,92507,null


### 1. Create one record for each customer with both addresses, one for each address_type
> [Documentation for PIVOT clause](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-qry-select-pivot)

In [0]:
SELECT *
 FROM (SELECT customer_id,
            address_type,
            address_line_1,
            city,
            state,
            postcode
        FROM gizmobox.bronze.v_addresses)
PIVOT (MAX(address_line_1) AS address_line_1,
       MAX(city) AS city,
       MAX(state) AS state,
       MAX(postcode) AS postcode
       FOR address_type IN ('shipping', 'billing')
       );

customer_id,shipping_address_line_1,shipping_city,shipping_state,shipping_postcode,billing_address_line_1,billing_city,billing_state,billing_postcode
1211,580 Samantha Lodge,New Christinaborough,Louisiana,4251,7938 Cervantes Mountain Suite 159,Oconnorhaven,Indiana,44129
1987,944 Smith Squares,Port Jennifer,Maryland,28646,3440 Beck Parkways,West Jennifer,Alaska,79766
2054,225 Lopez Bridge Suite 386,South Monica,Indiana,34134,890 Brandon Brook Suite 774,North Nicoleborough,Wyoming,15077
2141,098 Sellers Station,South Javierstad,Texas,59689,7451 Lisa Stravenue Suite 994,South Victor,Kansas,49835
2344,97619 Sanchez Prairie Suite 359,Kelleyfurt,Louisiana,64906,5008 Corey Landing Apt. 475,South Christinaburgh,New Mexico,3650
2639,5383 Albert Grove Apt. 804,North Destinyland,Illinois,60150,222 Dominique Springs Apt. 942,Colechester,Louisiana,96258
2703,5385 Paul Port Suite 963,Richardsonton,Florida,6432,7639 Derek Streets,New Mary,California,68610
3084,3337 Gregory Island,Mitchellbury,Tennessee,70603,78828 Montgomery Branch Suite 316,South Chelsea,Wisconsin,61054
3295,9569 Dennis Way Suite 047,North Amy,New York,17867,36789 Hatfield Landing,Haydenton,Tennessee,60950
3409,918 April Junctions,New Theresa,Indiana,46004,1294 Edward Inlet,North Alison,Indiana,82752


### 2. Write transformed data to the Silver schema 

In [0]:
CREATE TABLE gizmobox.silver.addresses
AS
SELECT *
 FROM (SELECT customer_id,
            address_type,
            address_line_1,
            city,
            state,
            postcode
        FROM gizmobox.bronze.v_addresses)
PIVOT (MAX(address_line_1) AS address_line_1,
       MAX(city) AS city,
       MAX(state) AS state,
       MAX(postcode) AS postcode
       FOR address_type IN ('shipping', 'billing')
       );

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM gizmobox.silver.addresses;

customer_id,shipping_address_line_1,shipping_city,shipping_state,shipping_postcode,billing_address_line_1,billing_city,billing_state,billing_postcode
1211,580 Samantha Lodge,New Christinaborough,Louisiana,4251,7938 Cervantes Mountain Suite 159,Oconnorhaven,Indiana,44129
1987,944 Smith Squares,Port Jennifer,Maryland,28646,3440 Beck Parkways,West Jennifer,Alaska,79766
2054,225 Lopez Bridge Suite 386,South Monica,Indiana,34134,890 Brandon Brook Suite 774,North Nicoleborough,Wyoming,15077
2141,098 Sellers Station,South Javierstad,Texas,59689,7451 Lisa Stravenue Suite 994,South Victor,Kansas,49835
2344,97619 Sanchez Prairie Suite 359,Kelleyfurt,Louisiana,64906,5008 Corey Landing Apt. 475,South Christinaburgh,New Mexico,3650
2639,5383 Albert Grove Apt. 804,North Destinyland,Illinois,60150,222 Dominique Springs Apt. 942,Colechester,Louisiana,96258
2703,5385 Paul Port Suite 963,Richardsonton,Florida,6432,7639 Derek Streets,New Mary,California,68610
3084,3337 Gregory Island,Mitchellbury,Tennessee,70603,78828 Montgomery Branch Suite 316,South Chelsea,Wisconsin,61054
3295,9569 Dennis Way Suite 047,North Amy,New York,17867,36789 Hatfield Landing,Haydenton,Tennessee,60950
3409,918 April Junctions,New Theresa,Indiana,46004,1294 Edward Inlet,North Alison,Indiana,82752
